## Understanding col function in Spark

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Understanding col function in Spark").getOrCreate()

24/02/24 14:39:21 WARN Utils: Your hostname, Qasim resolves to a loopback address: 127.0.1.1; using 172.30.54.131 instead (on interface eth0)
24/02/24 14:39:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/02/24 14:39:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
import datetime
users_list = [
    {
        'id': 1,
        'first_name': 'John',
        'last_name': 'Doe',
        'email': 'john.doe@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 54 7400113', 'PAK_Mobile': '+92 300 5638080'},
        'is_customer': True,
        'amount_paid': 123.45,
        'customer_from': datetime.date(2023, 1, 15),  # Date in YYYY-MM-DD format
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)  # Timestamp in YYYY-MM-DD HH:MM:SS format
    },
    {
        'id': 2,
        'first_name': 'Jane',
        'last_name': 'Smith',
        'email': 'jane.smith@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 99 7666666', 'PAK_Mobile': '+92 300 0000000'},
        'is_customer': True,
        'amount_paid': 0.0,
        'customer_from': datetime.date(2022, 11, 10),
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)
    },
    {
        'id': 3,
        'first_name': 'Jack',
        'last_name': 'Alpha',
        'email': 'jack.alpha@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 99 7666555'},
        'is_customer': False,
        'amount_paid': 0.0,
        'customer_from': datetime.date(2022, 11, 10),
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)
    }
    # Add more user dictionaries as needed
]

In [3]:
from pyspark.sql import Row

users_df = spark.createDataFrame(Row(**users) for users in users_list)
users_df

DataFrame[id: bigint, first_name: string, last_name: string, email: string, phone_numer: map<string,string>, is_customer: boolean, amount_paid: double, customer_from: date, last_updated_ts: timestamp]

In [4]:
users_df['id']

Column<'id'>

In [5]:
from pyspark.sql.functions import col
col('id')

Column<'id'>

In [6]:
cols = ['id', 'first_name', 'last_name']

In [7]:
users_df.select(*cols).show()

+---+----------+---------+
| id|first_name|last_name|
+---+----------+---------+
|  1|      John|      Doe|
|  2|      Jane|    Smith|
|  3|      Jack|    Alpha|
+---+----------+---------+



In [8]:
help(col)

Help on function col in module pyspark.sql.functions:

col(col: str) -> pyspark.sql.column.Column
    Returns a :class:`~pyspark.sql.Column` based on the given column name.
    
    .. versionadded:: 1.3.0
    
    .. versionchanged:: 3.4.0
        Supports Spark Connect.
    
    Parameters
    ----------
    col : str
        the name for the column
    
    Returns
    -------
    :class:`~pyspark.sql.Column`
        the corresponding column instance.
    
    Examples
    --------
    >>> col('x')
    Column<'x'>
    >>> column('x')
    Column<'x'>



In [9]:
user_id = col('id')

In [10]:
user_id

Column<'id'>

In [11]:
users_df.select(user_id).show()

+---+
| id|
+---+
|  1|
|  2|
|  3|
+---+



### Functions that can be used on top of the col object

- cast - select, filter, group by, orderBy
- asc - sort or orderBy
- contains - filter or where

In [12]:
users_df.select('id', 'first_name', 'customer_from').show()

+---+----------+-------------+
| id|first_name|customer_from|
+---+----------+-------------+
|  1|      John|   2023-01-15|
|  2|      Jane|   2022-11-10|
|  3|      Jack|   2022-11-10|
+---+----------+-------------+



In [13]:
users_df.select('id', 'first_name', 'customer_from').dtypes

[('id', 'bigint'), ('first_name', 'string'), ('customer_from', 'date')]

In [14]:
from pyspark.sql.functions import date_format

In [17]:
users_df.select(
    col('id'),
    col('first_name'),
    date_format('customer_from', 'yyyyMMdd').cast('int').alias('customer_from')
).show()

+---+----------+-------------+
| id|first_name|customer_from|
+---+----------+-------------+
|  1|      John|     20230115|
|  2|      Jane|     20221110|
|  3|      Jack|     20221110|
+---+----------+-------------+



In [18]:
users_df.select(
    col('id'),
    col('first_name'),
    date_format('customer_from', 'yyyyMMdd').cast('int').alias('customer_from')
).printSchema()

root
 |-- id: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- customer_from: integer (nullable = true)



In [19]:
users_df.select(
    col('id'),
    col('first_name'),
    date_format('customer_from', 'yyyyMMdd').cast('int').alias('customer_from')
).dtypes

[('id', 'bigint'), ('first_name', 'string'), ('customer_from', 'int')]

In [22]:
type(users_df['id'])

pyspark.sql.column.Column

In [24]:
type(user_id)

pyspark.sql.column.Column

In [25]:
users_df.select(
    'id',
    col('first_name'),
    date_format('customer_from', 'yyyyMMdd').cast('int').alias('customer_from')
).show()

+---+----------+-------------+
| id|first_name|customer_from|
+---+----------+-------------+
|  1|      John|     20230115|
|  2|      Jane|     20221110|
|  3|      Jack|     20221110|
+---+----------+-------------+



In [26]:
cols = [col('id'), date_format('customer_from', 'yyyyMMdd').cast('int').alias('customer_from')]
users_df.select(cols).show()

+---+-------------+
| id|customer_from|
+---+-------------+
|  1|     20230115|
|  2|     20221110|
|  3|     20221110|
+---+-------------+



In [27]:
cols = [col('id'), date_format('customer_from', 'yyyyMMdd').cast('int').alias('customer_from')]
users_df.select(*cols).show()

+---+-------------+
| id|customer_from|
+---+-------------+
|  1|     20230115|
|  2|     20221110|
|  3|     20221110|
+---+-------------+

